In [20]:
from pyspark.mllib.tree import GradientBoostedTrees
from pyspark.mllib.regression import LabeledPoint
from pyspark.sql import SparkSession
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.mllib.linalg import Vectors
from pyspark.sql.functions import when
import numpy as np

In [21]:
spark = SparkSession.builder.appName("GradientBoost").getOrCreate()
data = spark.read.csv("data.csv", header = True, inferSchema=True)

In [22]:
# M is encoded as 1 and B is encoded as 0
data = data.withColumn("label", 
                              when(data.diagnosis == "M", 1)
                              .otherwise(0))


feature_cols = [col for col in data.columns if col not in ['id', 'diagnosis', 'label']]
model_data = data.select(feature_cols + ['label'])

def create_labeled_point(row):
    features = [row[col] for col in feature_cols]
    return LabeledPoint(row.label, Vectors.dense(features))

rdd_data = model_data.rdd.map(create_labeled_point)

In [23]:
train_rdd, test_rdd = rdd_data.randomSplit([0.7, 0.3])

In [29]:
model = GradientBoostedTrees.trainClassifier(
    train_rdd,
    categoricalFeaturesInfo={},
    numIterations=20
)
predictions = model.predict(test_rdd.map(lambda x: x.features))
predictions_and_labels = test_rdd.map(lambda x: x.label).zip(predictions)

In [30]:
metrics = MulticlassMetrics(predictions_and_labels)
accuracy = metrics.accuracy
precision_1 = metrics.precision(1.0)
recall_1 = metrics.recall(1.0)
f1_1 = metrics.fMeasure(1.0)

In [ ]:
print(f"\n=== MODEL EVALUATION METRICS ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")


=== MODEL EVALUATION METRICS ===
Accuracy: 0.8968
Precision: 0.8448
Recall: 0.8750
F1-Score: 0.8596


In [ ]:
print(f"\n=== MODEL EVALUATION METRICS ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")


=== MODEL EVALUATION METRICS ===
Accuracy: 0.8968
Precision: 0.8448
Recall: 0.8750
F1-Score: 0.8596


In [33]:
confusion_matrix = metrics.confusionMatrix()
print(f"\n=== CONFUSION MATRIX ===")
print(confusion_matrix.toArray())


=== CONFUSION MATRIX ===
[[90.  9.]
 [ 7. 49.]]
